# Övning 4: Högre ordningens funktioner

I den här övningen arbetar vi med **högre ordningens funktioner**, framför allt `map`, `filter` och `foldr`, samt med sambandet mellan dessa och listbyggare.

En viktig idé är att vi ibland kan utelämna en parameter helt. Om vi till exempel har

```haskell
removeOdds xs = filter even xs
```

kan vi också skriva

```haskell
removeOdds = filter even
```

Båda definitionerna beskriver samma funktion. Den andra formen utnyttjar att `filter even` redan är en funktion som väntar på en lista.


## Lärardemo: Ta bort udda tal

Skriv en funktion som tar in en lista med heltal och tar bort samtliga udda tal i listan.


In [17]:
removeOdds :: [Int] -> [Int]
removeOdds xs = filter even xs


Line 2: Eta reduce
Found:
removeOdds xs = filter even xs
Why not:
removeOdds = filter even

In [18]:
removeOdds [1,2,3,4,5]


[2,4]

### Behöver vi verkligen `xs`?

I definitionen

```haskell
removeOdds xs = filter even xs
```

skickas `xs` bara vidare som argument till `filter even`.

Men `filter even` är redan en funktion av typen

```text
[Int] -> [Int]
```

så vi kan skriva definitionen direkt:

```haskell
removeOdds = filter even
```

Detta kallas ofta **point-free style**: vi beskriver funktionen utan att explicit namnge dess argument.


In [19]:
removeOdds' :: [Int] -> [Int]
removeOdds' = filter even


In [20]:
removeOdds' [1,2,3,4,5]


[2,4]

## Uppgift 2: Räkna element i en lista av listor

Skriv en funktion som tar in en lista av listor med heltal och räknar det **totala antalet element**.


In [37]:
numberOfElements :: [[Int]] -> Int
numberOfElements xs = sum (map length xs)


In [38]:
numberOfElements [[1,2],[],[3,4,5,6],[7],[8,9],[10]]


10

### Vad händer?

För listan

```text
[[1,2], [], [3,4,5,6], [7], [8,9], [10]]
```

gör `map length` först om varje lista till dess längd:

```text
map length [[1,2], [], [3,4,5,6], [7], [8,9], [10]]
= [2,0,4,1,2,1]
```

Därefter summerar `sum` resultatet:

```text
sum [2,0,4,1,2,1]
= 10
```

Vi har alltså en pipeline:

```text
listor  →  längder  →  summa
```


### Utan explicit parameter

Även här används `xs` bara som indata till ett sammansatt uttryck:

```haskell
numberOfElements xs = sum (map length xs)
```

Med funktionskomposition `(.)` kan samma funktion skrivas:

```haskell
numberOfElements = sum . map length
```

Läs uttrycket från höger: **beräkna först längden på varje lista, summera sedan resultaten**.


In [39]:
numberOfElements' :: [[Int]] -> Int
numberOfElements' = sum . map length


In [40]:
numberOfElements' [[1,2],[],[3,4,5,6],[7],[8,9],[10]]


10

## Uppgift 3: Från listbyggare till `map` och `filter`

Skriv om följande listbyggare med hjälp av `map` och `filter`:

```haskell
[x + 1 | x <- xs]
```

och

```haskell
[x + 4 | (x,y) <- xys, x + y < 5]
```


### Första uttrycket

Listbyggaren

```haskell
[x + 1 | x <- xs]
```

gör samma sak med **varje** element: adderar `1`.

Det är precis vad `map` uttrycker:


In [25]:
f :: Num a => [a] -> [a]
f = map (+1)


In [26]:
f [1,2,3]


[2,3,4]

### Andra uttrycket

Här händer två olika saker:

```haskell
[x + 4 | (x,y) <- xys, x + y < 5]
```

1. `filter` väljer först de par där `x + y < 5`.
2. `map` tar sedan första komponenten (`fst`) och adderar `4`.

Det kan skrivas:

```haskell
map ((+4) . fst) (filter (\(x,y) -> x+y < 5) xys)
```

eller, utan explicit parameter, som en komposition:

```haskell
g = map ((+4) . fst) . filter (\(x,y) -> x+y < 5)
```

Läs även här från höger: **filtrera först, transformera sedan**.


In [27]:
g :: (Num a, Ord a) => [(a,a)] -> [a]
g = map ((+4) . fst) . filter (\(x,y) -> x+y < 5)


In [28]:
g [(1,1),(1,2),(2,2),(2,3)]


[5,5,6]

## Uppgift 4: Från `filter` till listbyggare

Skriv om

```haskell
filter (>7) xs
```

med hjälp av en listbyggare.


In [29]:
greaterThanSeven :: (Num a, Ord a) => [a] -> [a]
greaterThanSeven xs = [x | x <- xs, x > 7]


In [30]:
greaterThanSeven [1,21,3]


[21]

Här motsvarar villkoret i listbyggaren precis predikatet som används av `filter`:

```text
filter (>7) xs
        ↓
[x | x <- xs, x > 7]
```


## Uppgift 5: Skriv `map` med `foldr`

Skriv om `map f` med hjälp av `foldr`.


In [31]:
map' :: (a -> b) -> [a] -> [b]
map' f = foldr (\x xs -> f x : xs) []


Line 2: Use map
Found:
foldr (\ x xs -> f x : xs) []
Why not:
map (\ x -> f x)

### Hur fungerar det?

`foldr` ersätter successivt listans `(:)` med den funktion vi anger och `[]` med startvärdet.

För

```haskell
map' (+1) [1,2,3]
```

kan vi tänka:

```text
foldr (\x xs -> (x+1) : xs) [] [1,2,3]

= (1+1) : ((2+1) : ((3+1) : []))
= 2 : (3 : (4 : []))
= [2,3,4]
```

Det viktiga är att `foldr` här **bygger en ny lista**. För varje element `x` lägger vi `f x` först i den lista som återstår:

```haskell
\x xs -> f x : xs
```

Startvärdet är den tomma listan `[]`.


In [32]:
map' (+1) [1,2,3]


[2,3,4]

---

## Sammanfattning

I övningen har vi sett att samma listbearbetning ofta kan uttryckas på flera sätt:

- `map` transformerar varje element.
- `filter` väljer vilka element som ska vara kvar.
- Listbyggare kan ofta uttrycka samma sak som kombinationer av `map` och `filter`.
- `foldr` kan användas för att bygga upp många andra listfunktioner.
- En parameter behöver inte namnges om den bara skickas vidare till en annan funktion.

Till exempel är

```haskell
removeOdds xs = filter even xs
```

och

```haskell
removeOdds = filter even
```

två sätt att definiera samma funktion.
